# Tesla (TSLA) Stock Analysis: Moving Averages & Statistics

This notebook demonstrates how to load Tesla (`TSLA`) stock historical data, calculate moving averages, and analyze standard statistical metrics.

> [!NOTE]
> **Why did `pandas_datareader` fail?**
> Yahoo Finance has deprecated/changed the backend endpoint used by `pandas_datareader`'s scraper, causing a `RemoteDataError`. 
> The modern and robust solution is to use the native **`yfinance`** library directly, which fetches data reliably.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a premium visual style for our data visualization
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

## 1. Fetching Tesla Stock Data

We use `yf.download()` to retrieve historical stock data for `TSLA`. We specify `auto_adjust=False` to retain both the standard `Close` and the split/dividend-adjusted `Adj Close` column, and `multi_level_index=False` to return a flat, easy-to-use DataFrame.

In [ ]:
# Fetching TSLA stock data from the last 5 years
df_tesla = yf.download(
    tickers='TSLA',
    period='5y',
    auto_adjust=False,
    multi_level_index=False
)

df_tesla.head()

## 2. Descriptive Statistics

Let's check the shape, info, and summary statistics of the dataset.

In [ ]:
# Get summary statistics
df_tesla.describe()

## 3. Simple Moving Averages (SMA)

Simple Moving Averages help identify trends by smoothing out daily price fluctuations. We'll calculate:
- **20-day SMA** (Short-term trend)
- **50-day SMA** (Medium-term trend)
- **200-day SMA** (Long-term trend)

In [ ]:
# Calculate SMAs
df_tesla['SMA_20'] = df_tesla['Adj Close'].rolling(window=20).mean()
df_tesla['SMA_50'] = df_tesla['Adj Close'].rolling(window=50).mean()
df_tesla['SMA_200'] = df_tesla['Adj Close'].rolling(window=200).mean()

# Display recent rows
df_tesla[['Adj Close', 'SMA_20', 'SMA_50', 'SMA_200']].tail()

### Plotting Moving Averages

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(df_tesla.index, df_tesla['Adj Close'], label='TSLA Adj Close', color='#1f77b4', alpha=0.5)
plt.plot(df_tesla.index, df_tesla['SMA_20'], label='20-Day SMA', color='#ff7f0e', linewidth=1.5)
plt.plot(df_tesla.index, df_tesla['SMA_50'], label='50-Day SMA', color='#2ca02c', linewidth=1.8)
plt.plot(df_tesla.index, df_tesla['SMA_200'], label='200-Day SMA', color='#d62728', linewidth=2.0)

plt.title('Tesla (TSLA) Stock Price & Simple Moving Averages (SMA)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Adjusted Close Price (USD)', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 4. Bollinger Bands

Bollinger Bands consist of a middle band (usually a 20-day SMA) and two outer bands that are standard deviations away from the middle band. They help identify overbought and oversold conditions.

In [ ]:
# Middle Band = 20 SMA
df_tesla['BB_Middle'] = df_tesla['Adj Close'].rolling(window=20).mean()
# Standard Deviation
std_dev = df_tesla['Adj Close'].rolling(window=20).std()
# Upper & Lower Bands (2 standard deviations away)
df_tesla['BB_Upper'] = df_tesla['BB_Middle'] + (std_dev * 2)
df_tesla['BB_Lower'] = df_tesla['BB_Middle'] - (std_dev * 2)

# Plot the Bollinger Bands for the last year for high detail
df_recent = df_tesla.tail(252)  # Approximately 1 year of trading days

plt.figure(figsize=(15, 8))
plt.plot(df_recent.index, df_recent['Adj Close'], label='TSLA Adj Close', color='#333333', linewidth=1.5)
plt.plot(df_recent.index, df_recent['BB_Middle'], label='Bollinger Middle (20 SMA)', color='#007acc', linestyle='--')
plt.plot(df_recent.index, df_recent['BB_Upper'], label='Bollinger Upper Band', color='#2ca02c', alpha=0.7)
plt.plot(df_recent.index, df_recent['BB_Lower'], label='Bollinger Lower Band', color='#d62728', alpha=0.7)

# Fill the space between bands
plt.fill_between(df_recent.index, df_recent['BB_Lower'], df_recent['BB_Upper'], color='#007acc', alpha=0.1, label='BB Region')

plt.title('Tesla (TSLA) Stock Price & Bollinger Bands (Last 1 Year)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()